## Imports

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.ticker import MultipleLocator, FixedLocator, FixedFormatter

KeyboardInterrupt: 

KeyboardInterrupt: 

## Leer CSV

In [ ]:
# Leer CSV
# 📂 Leer CSV - versión simple y robusta
ruta_csv = input("Introduce la ruta al archivo CSV: ").strip().strip('"').strip("'") # Elimina espacios y comillas si las hay

df = pd.read_csv(ruta_csv)

## Procesamiento y salida gráfica

In [ ]:
# =============================================================================
# UTILIDADES
# =============================================================================

def formatear_pk(pk_km):
    """
    Convierte PK (km float) a notación K+MMM (ej. 91.740 -> 91+740).
    Controla carry por redondeo (999.6 m -> +1 km).
    """
    pk_km = np.asarray(pk_km, dtype=float)
    km = np.floor(pk_km).astype(int)
    m = np.round((pk_km - km) * 1000).astype(int)

    carry = m // 1000
    km = km + carry
    m = m % 1000
    return [f"{k}+{mm:03d}" for k, mm in zip(km, m)]


def determinar_salto_y(max_y, min_y):
    rango = max_y - min_y
    if rango <= 100:   return 10
    if rango <= 250:   return 25
    if rango <= 500:   return 50
    if rango <= 1000:  return 100
    return 250


def elegir_step(rango, candidatos, target_labels=10):
    """
    Selecciona un paso "agradable" aproximando el nº objetivo de etiquetas.
    """
    if rango <= 0:
        return candidatos[0]
    step_obj = rango / max(1, (target_labels - 1))
    for s in candidatos:
        if s >= step_obj:
            return s
    return candidatos[-1]


def unique_mean(x_base, y_base):
    """
    Ordena por x_base y colapsa duplicados promediando y_base.
    Devuelve (x_u, y_u) con x_u sin duplicados y creciente.
    """
    x = np.asarray(x_base, dtype=float)
    y = np.asarray(y_base, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if x.size == 0:
        return x, y

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    xu, inv = np.unique(x, return_inverse=True)
    y_sum = np.bincount(inv, weights=y)
    y_cnt = np.bincount(inv)
    yu = y_sum / np.maximum(y_cnt, 1)
    return xu, yu


def interp_extrap_pk_to_x(pk_new, pk_base, x_base):
    """
    Transformación PK(km) -> X(m) robusta:
      - deduplicación en PK (media)
      - imposición de monotonicidad X no decreciente con PK
      - interpolación lineal interna
      - extrapolación en extremos con el primer/último tramo con incremento real en X
    """
    pk_new = np.asarray(pk_new, dtype=float)
    pk_u, x_u = unique_mean(pk_base, x_base)

    if pk_u.size == 0:
        return np.full_like(pk_new, np.nan, dtype=float)
    if pk_u.size == 1:
        return np.full_like(pk_new, x_u[0], dtype=float)

    # Monotonicidad (evita extrapolación "hacia atrás" por ruido)
    x_u = np.maximum.accumulate(x_u)

    # Interpolación interna
    x_new = np.interp(pk_new, pk_u, x_u)

    dx = np.diff(x_u)
    idx = np.where(dx > 1e-9)[0]

    # Pendiente inferior
    if idx.size > 0:
        i0 = idx[0]
        denom0 = pk_u[i0 + 1] - pk_u[i0]
        m0 = 0.0 if denom0 == 0 else (x_u[i0 + 1] - x_u[i0]) / denom0
    else:
        m0 = 0.0

    lo = pk_new < pk_u[0]
    x_new[lo] = x_u[0] + (pk_new[lo] - pk_u[0]) * m0

    # Pendiente superior
    if idx.size > 0:
        i1 = idx[-1]
        denom1 = pk_u[i1 + 1] - pk_u[i1]
        m1 = 0.0 if denom1 == 0 else (x_u[i1 + 1] - x_u[i1]) / denom1
    else:
        m1 = 0.0

    hi = pk_new > pk_u[-1]
    x_new[hi] = x_u[-1] + (pk_new[hi] - pk_u[-1]) * m1

    return x_new


def construir_ticks(x_m, pk_km=None, modo="pk", target_labels=10):
    """
    Construye ticks y etiquetas para el eje X.
      - modo="pk": ticks en dominio PK (km) -> posiciones X (m)
      - modo="m" : ticks en metros (con autoetiquetado m/km por rango)
    """
    x_m = np.asarray(x_m, dtype=float)

    if modo == "pk":
        pk_km = np.asarray(pk_km, dtype=float)

        pk_u, x_u = unique_mean(pk_km, x_m)
        vmin, vmax = np.nanmin(pk_u), np.nanmax(pk_u)
        rango = vmax - vmin

        candidatos = [0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50]  # km
        step = elegir_step(rango, candidatos, target_labels=target_labels)

        start = np.floor(vmin / step) * step            # incluye múltiplo anterior
        end = np.ceil(vmax / step) * step               # salta al siguiente múltiplo
        ticks_pk = np.arange(start, end + 1e-12, step)

        tick_positions = interp_extrap_pk_to_x(ticks_pk, pk_u, x_u)
        tick_labels = formatear_pk(ticks_pk)
        return tick_positions, tick_labels

    elif modo == "m":
        vmin, vmax = np.nanmin(x_m), np.nanmax(x_m)
        rango = vmax - vmin

        candidatos = [10, 25, 50, 100, 200, 250, 500, 1000, 2000, 5000, 10000]
        step = elegir_step(rango, candidatos, target_labels=target_labels)

        start = np.floor(vmin / step) * step
        end = np.ceil(vmax / step) * step
        ticks_m = np.arange(start, end + 1e-12, step)

        tick_positions = ticks_m

        # ✅ Decisión de unidades por rango (umbral acordado: 2.5 km)
        usar_km = rango >= 2500

        if usar_km:
            ticks_km = ticks_m / 1000.0
            step_km = step / 1000.0
            if np.isclose(step_km, round(step_km), atol=1e-12):
                tick_labels = [f"{int(round(t))} km" for t in ticks_km]
            else:
                tick_labels = [f"{t:.1f} km" for t in ticks_km]
        else:
            tick_labels = [f"{int(t)} m" for t in ticks_m]

        return tick_positions, tick_labels

    else:
        raise ValueError("modo debe ser 'pk' o 'm'")


def aplicar_ticks(ax, tick_positions, tick_labels, x_data=None,
                 extend_left=True, extend_right=True, xpad_frac=0.02):
    """
    Aplica ticks fijos al eje X y ajusta límites para incluir ticks extrapolados.
    """
    pos = np.asarray(tick_positions, dtype=float)
    lab = np.asarray(tick_labels, dtype=object)

    # Orden y deduplicación por tolerancia (flotantes)
    order = np.argsort(pos)
    pos, lab = pos[order], lab[order]
    if pos.size > 1:
        keep = np.ones(pos.size, dtype=bool)
        keep[1:] = np.abs(np.diff(pos)) > 1e-6
        pos, lab = pos[keep], lab[keep]

    ax.xaxis.set_major_locator(FixedLocator(pos))
    ax.xaxis.set_major_formatter(FixedFormatter(lab))

    ax.tick_params(axis="x", rotation=45, pad=10)
    for t in ax.get_xticklabels():
        t.set_ha("right")
        t.set_rotation_mode("anchor")
        t.set_clip_on(False)

    ax.minorticks_off()

    if x_data is None:
        xmin_d, xmax_d = np.nanmin(pos), np.nanmax(pos)
    else:
        xd = np.asarray(x_data, dtype=float)
        xmin_d, xmax_d = np.nanmin(xd), np.nanmax(xd)

    xmin = min(xmin_d, np.nanmin(pos)) if extend_left else xmin_d
    xmax = max(xmax_d, np.nanmax(pos)) if extend_right else xmax_d

    span = (xmax - xmin) if xmax > xmin else 1.0
    ax.set_xlim(xmin, xmax + xpad_frac * span)


# =============================================================================
# PREPROCESADO ROBUSTO (PK opcional)
# =============================================================================

required_cols = ["Cota_SUAV", "Dist_Origen_metros"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas obligatorias: {missing}")

pk_col = "m_field_PK_KM"

# Pregunta al usuario ANTES de exigir PK
usar_pk = input("¿Etiquetar eje X como PK? (s/n): ").strip().lower() == "s"
modo_x = "pk" if usar_pk else "m"

# Limpieza mínima común
df2 = df.dropna(subset=required_cols).copy()

# Si se pide PK, validar existencia; si no, fallback a metros
if modo_x == "pk" and pk_col not in df2.columns:
    print(f"⚠️ No existe la columna '{pk_col}'. Se etiquetará en distancia (metros o km).")
    modo_x = "m"

# Si finalmente se usa PK, eliminar NaN PK
if modo_x == "pk":
    df2 = df2.dropna(subset=[pk_col])

# Ordenar por X y colapsar duplicados en X (evita diagonales en fill_between)
df2 = df2.sort_values("Dist_Origen_metros")

agg = {"Cota_SUAV": "mean"}
if "SLOPE" in df2.columns:
    agg["SLOPE"] = "mean"
if pk_col in df2.columns:
    agg[pk_col] = "mean"

df2 = df2.groupby("Dist_Origen_metros", as_index=False).agg(agg)

# Arrays finales
x = df2["Dist_Origen_metros"].to_numpy(dtype=float)
y = df2["Cota_SUAV"].to_numpy(dtype=float)
slope = df2["SLOPE"].to_numpy() if "SLOPE" in df2.columns else np.full_like(x, np.nan, dtype=float)

pk_field = df2[pk_col].to_numpy(dtype=float) if (modo_x == "pk") else None

salto_y = determinar_salto_y(np.nanmax(y), np.nanmin(y))

# Ticks
tick_positions, tick_labels = construir_ticks(x, pk_field, modo=modo_x, target_labels=10)


# =============================================================================
# FIGURA 1: PERFIL
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

ax.set_facecolor((0.7, 0.7, 0.7, 0.15))
ax.fill_between(x, y, where=~np.isnan(y), color="#1b5e20", alpha=0.8, interpolate=True)
ax.plot(x, y, color="#0d3b12", linewidth=1)

ax.set_ylabel("Altura (m)")
ax.set_xlabel("Distancia")

ax.grid(True, axis="both", which="major",
        linestyle='-',
        linewidth=0.4,
        color='gray')

ax.yaxis.set_major_locator(MultipleLocator(salto_y))
aplicar_ticks(ax, tick_positions, tick_labels, x_data=x,
             extend_left=True, extend_right=True, xpad_frac=0.02)

fig.patch.set_alpha(0.0)

plt.title("Perfil Longitudinal")
plt.tight_layout()
plt.show()


# =============================================================================
# FIGURA 2: PERFIL + PENDIENTE
# =============================================================================

fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_facecolor((0.7, 0.7, 0.7, 0.15))
ax1.fill_between(x, y, where=~np.isnan(y), color="#1b5e20", alpha=0.8, interpolate=True)
ax1.plot(x, y, color="#0d3b12", linewidth=1)

ax1.set_ylabel("Altura (m)")
ax1.set_xlabel("Distancia")

ax1.grid(True, axis="both", which="major",
         linestyle='-',
         linewidth=0.4,
         color='gray')

ax1.yaxis.set_major_locator(MultipleLocator(salto_y))
aplicar_ticks(ax1, tick_positions, tick_labels, x_data=x,
             extend_left=True, extend_right=True, xpad_frac=0.02)

ax2 = ax1.twinx()
ax2.plot(x, slope, color="black", linewidth=1.2, linestyle="--")
ax2.set_ylabel("Pendiente (%)", color="black")
ax2.tick_params(axis="y", labelcolor="black")

fig.patch.set_alpha(0.0)

plt.title("Perfil Longitudinal con Pendiente (%)")
plt.tight_layout()
plt.show()